In [1]:
# ===== Cell 1: 封装天气查询函数（心知天气 API）=====
import requests

API_KEY = os.getenv('SENIVERSE_API_KEY', '')  # 心知天气 API Key
BASE_URL = 'https://api.seniverse.com/v3/weather/now.json'


def ask_weather(city):
    '''查询指定城市的实时天气

    参数:
        city: 城市名，如 '黄冈'、'北京'、'上海'
    返回:
        天气描述字符串；失败时返回错误提示
    '''
    # 用 params 字典传参，requests 会自动对中文城市名做 URL 编码
    params = {
        'key': API_KEY,
        'location': city,
        'language': 'zh-Hans',
        'unit': 'c',        # c=摄氏度
    }

    try:
        resp = requests.get(BASE_URL, params=params, timeout=10)
    except requests.RequestException as e:
        return f'网络请求失败: {e}'

    # 处理 API 返回的错误
    if resp.status_code != 200:
        try:
            err = resp.json()
            return f'查询失败: {err.get("status", resp.status_code)}'
        except Exception:
            return f'查询失败: HTTP {resp.status_code}'

    data = resp.json()
    if 'results' not in data or not data['results']:
        return f'未找到城市「{city}」的天气信息'

    result = data['results'][0]
    location = result['location']['name']
    now = result['now']
    updated = result['last_update'].replace('T', ' ').split('+')[0]

    return (f'{location}当前天气: {now["text"]}, '
            f'气温 {now["temperature"]}°C (更新于 {updated})')



In [2]:
# ===== Cell 2: 测试天气查询函数 =====
print(ask_weather('黄冈'))
print(ask_weather('北京'))



黄冈当前天气: 晴, 气温 20°C (更新于 2026-09-11 01:34:36)
北京当前天气: 晴, 气温 15°C (更新于 2026-09-11 01:35:08)


In [3]:
# ===== Cell 3: 把普通函数封装成 Langchain 工具 =====
# 什么是"工具(Tool)"？
#   工具就是一个带说明书的函数。@tool 装饰器会读取函数名、docstring 和类型注解，
#   自动生成一份"工具说明书"（名称 + 用途 + 参数）。
#   大模型自己不会查天气，但它能看懂这份说明书，在需要时决定调用工具、并填好参数。
from langchain_core.tools import tool


@tool
def weather_tool(city: str) -> str:
    '''查询中国某个城市的实时天气状况，包括天气现象和气温。
    当用户询问某个城市的天气、温度、冷不冷、热不热、下不下雨时使用此工具。

    参数:
        city: 城市名称，例如 "黄冈"、"北京"、"上海"
    '''
    # 工具内部直接调用 Cell 1 写好的普通函数
    return ask_weather(city)


print('Langchain 工具封装完成')
print(f'工具名称: {weather_tool.name}')
print(f'工具描述: {weather_tool.description}')
print(f'参数定义: {weather_tool.args}')



Langchain 工具封装完成
工具名称: weather_tool
工具描述: 查询中国某个城市的实时天气状况，包括天气现象和气温。
    当用户询问某个城市的天气、温度、冷不冷、热不热、下不下雨时使用此工具。

    参数:
        city: 城市名称，例如 "黄冈"、"北京"、"上海"
参数定义: {'city': {'title': 'City', 'type': 'string'}}


In [4]:
# ===== Cell 4: 创建 Agent（智能体）=====
# 什么是 Agent？
#   普通问答：用户问题 -> 大模型直接回答（大模型不知道实时天气，只能瞎编或说不知道）
#   Agent：   用户问题 -> 大模型【决定是否调用工具】-> 调用天气工具拿到真实数据
#             -> 大模型【根据工具结果组织语言】-> 最终回答
#   大模型在这里像一个"会思考的调度员"，工具是它手中的"武器"。
#
# LangChain 1.x 用 create_agent() 创建智能体（底层是 LangGraph 图结构）：
#   model=         负责思考和决策的大模型（这里用商汤 SenseNova 付费模型）
#   tools=         交给它使用的工具列表
#   system_prompt= 设定智能体的人设/行为规则
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 1) 接入商汤 SenseNova 付费模型（与 01.Langchain.ipynb 中是同一个模型）
#    ChatOpenAI 是"OpenAI 兼容接口"的通用入口，改 base_url 就能接入各家模型
llm = ChatOpenAI(
    model='sensenova-6.8-flash-lite',
    base_url='https://token.sensenova.cn/v1',
    api_key=os.getenv('SENSENOVA_API_KEY', ''),
    temperature=0.3,   # 温度低一些，回答更稳定
)

# 2) 创建智能体：把"大脑(模型)"和"双手(工具)"组装起来
agent = create_agent(
    model=llm,
    tools=[weather_tool],
    system_prompt=(
        '你是一个贴心的天气助手。'
        '当用户询问天气时，必须先调用天气工具查询真实数据，再根据结果用中文简洁回答；'
        '不要编造天气信息。'
    ),
)

print('Agent 创建完成！它现在具备了查询实时天气的能力。')



Agent 创建完成！它现在具备了查询实时天气的能力。


In [11]:
# ===== Cell 5: 运行 Agent，观察它自动调用工具 =====
# 调用方式：agent.invoke({'messages': [用户消息]})
# 返回值是一个字典，其中 'messages' 记录了完整的思考/行动过程，
# 最后一条消息就是给用户的最终回答。

def chat(user_input):
    '''向 Agent 发送一句话，并打印最终回答'''
    result = agent.invoke({
        'messages': [{'role': 'user', 'content': user_input}]
    })
    answer = result['messages'][-1].content.strip()  # 最后一条=最终回答, strip去掉模型多输出的空行
    print(f'我: {user_input}')
    print(f'助手: {answer}\n')
    return result

# 问题1：问天气 —— Agent 应该自动调用 weather_tool
r1 = chat('深圳今天天气怎么样？')

# 问题2：换个城市
r2 = chat('武汉现在冷不冷？')

# 问题3：闲聊 —— Agent 应该判断"不需要工具"，直接回答
r3 = chat('你好，你是谁？')



我: 深圳今天天气怎么样？
助手: 深圳今天天气晴朗 ☀️，气温 26°C，体感舒适，适合外出活动。

（数据更新于 2026-09-11 01:44）

我: 武汉现在冷不冷？
助手: 武汉现在是晴天，气温 20°C，体感比较舒适，不算冷。不过早晚温差可能稍大，出门可以备一件薄外套以防着凉。

我: 你好，你是谁？
助手: 你好！我是**商量**（SenseNova），由商汤科技（SenseTime）开发的日日新融合模态大模型。我可以帮你处理文字、图像等多模态任务，比如回答问题、写作、编程、分析图片等。有什么可以帮你的吗？😊



In [18]:
chat('黄冈天气怎么样，有什么要注意的吗？')

我: 黄冈天气怎么样，有什么要注意的吗？
助手: 黄冈当前天气情况如下：

☀️ **天气：晴**
🌡️ **气温：20°C**
（数据更新于 2026-09-11 01:35）

---

**温馨提示：**

1. **早晚温差**：20°C 的气温体感舒适，但早晚可能偏凉，建议出门时带一件薄外套。
2. **防晒**：晴天紫外线较强，白天外出注意涂抹防晒霜、戴帽子或墨镜。
3. **补水**：天气晴朗干燥，注意多喝水，保持身体水分。
4. **穿衣建议**：适合穿长袖T恤、薄衬衫等轻便舒适的衣物。

总体来说，今天黄冈天气不错，适合户外活动，享受好天气吧！😊



{'messages': [HumanMessage(content='黄冈天气怎么样，有什么要注意的吗？', additional_kwargs={}, response_metadata={}, id='cf31ca27-ae79-4987-ad1d-acff86a5c467'),
  AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 452, 'total_tokens': 499, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 18, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'sensenova-6.8-flash-lite', 'system_fingerprint': None, 'id': '3213ec4b-9436-481a-a415-d2087781b9b6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a08c78-00c1-7ce2-bfee-44f4b14ab138-0', tool_calls=[{'name': 'weather_tool', 'args': {'city': '黄冈'}, 'id': 'call_853cd88a079a48deb8c674f7', 'type': 'tool_call'}], invalid_to

In [16]:
# ===== Cell 6: 拆解 Agent 的完整思考过程（进阶理解）=====
# Agent 的每条消息都有一个 type，代表它的"一个动作"：
#   human    = 用户说的话
#   ai       = 大模型的回复；如果其中带 tool_calls，说明它决定调用工具
#   tool     = 工具执行后返回的结果
# 打印出来就能清楚看到：大模型如何"思考 -> 调工具 -> 看结果 -> 回答"
for msg in r4['messages']:
    if msg.type == 'human':
        print(f'[用户] {msg.content}')
    elif msg.type == 'ai':
        if getattr(msg, 'tool_calls', None):
            # 大模型决定调用工具
            for call in msg.tool_calls:
                print(f'[大模型决策] 调用工具「{call["name"]}」, 参数={call["args"]}')
        else:
            print(f'[大模型回答] {msg.content.strip()}')
    elif msg.type == 'tool':
        print(f'[工具返回] {msg.content}')



[用户] 黄冈天气怎么样，有什么要注意的吗？
[大模型决策] 调用工具「weather_tool」, 参数={'city': '黄冈'}
[工具返回] 黄冈当前天气: 晴, 气温 20°C (更新于 2026-09-11 01:35:08)
[大模型回答] 黄冈当前天气情况如下：

☀️ **天气：晴**
🌡️ **气温：20°C**
（数据更新于 2026-09-11 01:35）

**注意事项：**

1. **早晚温差**：20°C 的气温体感舒适，但早晚可能偏凉，建议出门带一件薄外套。
2. **防晒**：晴天紫外线较强，白天外出注意做好防晒措施（防晒霜、遮阳伞等）。
3. **补水**：晴朗天气容易让人忽略补水，记得多喝水。
4. **穿衣建议**：适合穿长袖T恤或薄衬衫，舒适又保暖。

总体来说，今天黄冈天气不错，适合外出活动！😊
